# DriverGuard · Обучение модели
**Четыре класса: телефон, сигарета, ремень пристёгнут / не пристёгнут.**

1. Меню **Среда выполнения → Сменить среду выполнения → GPU T4** (или другая доступная GPU).
2. Выполните ячейки по порядку. Для автоматической загрузки нужен ваш ключ Roboflow: его вводят в скрытое поле, не в чат. Альтернатива — загрузить ZIP набора в формате YOLOv8.
3. После шага 4 посмотрите примеры разметки. Если всё соответствует подписям, запустите обучение.
4. В конце скачайте **DriverGuard_Training_Result.zip** и передайте его для подключения и обновления результатов презентации.

Обучение выполняется в вашей сессии Colab. До её завершения новых обученных весов и измеренной точности нет.
Этот детектор предназначен для Python-устройства DriverGuard. Телефонный сценарий закрытия глаз использует MediaPipe и работает независимо от этого обучения.

## 1. Среда и сохранение
По умолчанию результаты и последние веса каждой эпохи копируются в вашу папку `MyDrive/DriverGuard`.
Colab запросит доступ к Drive. Если он не нужен, установите `SAVE_TO_DRIVE = False`; при сбросе сессии локальные файлы исчезнут.

In [ ]:
import os, sys, subprocess, json, shutil, time, hashlib, platform, ast
from pathlib import Path
from datetime import datetime, timezone
os.environ['WANDB_MODE'] = 'disabled'
os.environ['YOLO_AUTOINSTALL'] = 'false'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics==8.3.253', 'roboflow==1.2.16', 'onnx>=1.16,<1.20',
    'onnxruntime>=1.18,<2', 'PyYAML>=6,<7', 'Pillow>=10', 'matplotlib'])
import torch
assert torch.cuda.is_available(), 'Нужна GPU: Среда выполнения → Сменить среду выполнения → GPU.'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
WORK = Path('/content/DriverGuard') / RUN_ID
WORK.mkdir(parents=True, exist_ok=False)
SAVE_TO_DRIVE = True
BACKUP = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP = Path('/content/drive/MyDrive/DriverGuard') / RUN_ID
    BACKUP.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Папка запуска:', WORK)
print('Резервная копия:', BACKUP or 'отключена')

## 2. Данные
Выбран [PFE v1](https://universe.roboflow.com/pfe-lcrvh/pfe-scklc): страница проекта указывает 21 142 изображения и лицензию CC BY 4.0.
Это число на странице, не гарантированный размер конкретного экспорта. Реальные количества появятся после загрузки и проверки.
Автор набора: PFE. Исходные `face` и `food` исключаются; четыре нужных класса приводятся к именам DriverGuard.

Вариант A: `DATA_MODE = 'roboflow'`. Получите Private API Key в настройках своего Roboflow и вставьте в скрытое поле, либо сохраните в секретах Colab как `ROBOFLOW_API_KEY`.
Вариант B: выберите **Download Dataset → YOLOv8 → ZIP** на странице версии; установите `DATA_MODE = 'upload'` и загрузите ZIP при запросе.
Ключ не сохраняется в итоговый архив. Ошибки загрузки выводятся без содержимого ответа с возможным токеном.

In [ ]:
DATA_MODE = 'roboflow'  # 'roboflow' или 'upload'
RAW = WORK / 'raw'
if DATA_MODE == 'roboflow':
    from getpass import getpass
    from roboflow import Roboflow
    from google.colab import userdata
    import contextlib, io
    try:
        key = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        print('Вставьте ключ в поле ниже и нажмите Enter — без Enter загрузка не начнётся.', flush=True)
        key = getpass('Roboflow Private API Key (ввод скрыт): ')
    print('После вставки ключа обязательно нажмите Enter в поле ввода.', flush=True)
    assert key and key.strip(), 'Введите ключ или используйте DATA_MODE = upload.'
    print('Скачиваем данные. После загрузки начнётся отдельная проверка изображений.', flush=True)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            rf = Roboflow(api_key=key.strip())
            downloaded = rf.workspace('pfe-lcrvh').project('pfe-scklc').version(1).download('yolov8', location=str(RAW))
    except Exception:
        raise RuntimeError('Не удалось скачать PFE v1. Проверьте доступ/ключ, либо загрузите YOLOv8 ZIP через DATA_MODE = upload.') from None
    finally:
        key = None
        if 'rf' in globals(): del rf
else:
    assert DATA_MODE == 'upload', 'Неизвестный DATA_MODE'
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    assert len(archives) == 1, 'Загрузите один ZIP в формате YOLOv8.'
    RAW.mkdir()
    with zipfile.ZipFile(archives[0]) as z:
        assert sum(x.file_size for x in z.infolist()) < 15 * 1024**3, 'ZIP больше 15 ГБ.'
        for item in z.infolist():
            target = (RAW/item.filename).resolve()
            assert target.is_relative_to(RAW.resolve()), 'Некорректный путь в ZIP.'
        z.extractall(RAW)
    uploaded.clear()
configs = list(RAW.rglob('data.yaml'))
assert len(configs) == 1, 'В экспорте должен быть ровно один data.yaml.'
RAW = configs[0].parent
print('Данные загружены; теперь проверяем разметку и выборки.')

## 3. Проверка и подготовка — исправлено 03.10.2026
Проверка показывает количество обработанных изображений. Три известных файла PFE v1 с неподходящим форматом разметки и связанные варианты исключаются целиком, с записью причин в dataset_audit.json. Исходные файлы сохраняются. Неожиданные ошибки по-прежнему останавливают проверку. Все четыре класса должны остаться в каждой выборке.

Повторный запуск создаёт отдельную папку prepared_fixed_… и не удаляет предыдущие данные. Разделение train/valid/test сохраняется; независимость людей и соседних кадров не подтверждена.


In [ ]:
UTILITY_SOURCE = '"""Validate and remap a Roboflow YOLO export without mixing its held-out splits."""\nfrom __future__ import annotations\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nimport hashlib\nimport json\nimport math\nimport re\nimport shutil\nimport time\nimport yaml\nfrom PIL import Image\n\nCLASSES = ["phone", "cigarette", "seatbelt_on", "seatbelt_off"]\nALIASES = {"phone": 0, "cell phone": 0, "mobile phone": 0,\n           "cigarette": 1, "seatbelt": 2, "seat belt": 2, "seatbelt on": 2,\n           "seatbelt off": 3, "no seatbelt": 3, "no seat belt": 3}\nSOURCE = {"title": "PFE", "author": "PFE (Roboflow workspace pfe-lcrvh)",\n          "url": "https://universe.roboflow.com/pfe-lcrvh/pfe-scklc",\n          "version": 1, "license_as_listed": "CC BY 4.0",\n          "changes": "Four-class remapping; audited exclusion of three malformed PFE v1 labels; duplicate quarantine; source splits preserved."}\n\n# These three names are from the user\'s Colab validation report, not a blanket\n# permission to ignore malformed annotations in arbitrary datasets.\nKNOWN_PFE_V1_INVALID_LABELS = frozenset({\n    "train/labels/2183_jpg.rf.0631bd6ee34b77570ec9d796c14783d0.txt",\n    "train/labels/2183_jpg.rf.0641f1910972e6ae139d5dc5f43e22e5.txt",\n    "train/labels/2183_jpg.rf.ee7e9cef2ffa42c44592cfcbb9a1168d.txt",\n})\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for part in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(part)\n    return h.hexdigest()\n\ndef normalize(name):\n    return " ".join(str(name).lower().replace("_", " ").replace("-", " ").split())\n\ndef parse_labels(path, mapping):\n    if not path.is_file():\n        raise ValueError(f"Missing label file (not assumed to be a negative): {path}")\n    output, ignored = [], 0\n    for number, line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), 1):\n        if not line.strip():\n            continue\n        parts = line.split()\n        if len(parts) != 5:\n            raise ValueError(f"Expected detection boxes, 5 fields: {path}:{number}")\n        values = list(map(float, parts))\n        cls, x, y, w, h = values\n        if not all(map(math.isfinite, values)) or cls != int(cls) or int(cls) not in mapping:\n            raise ValueError(f"Invalid class/number: {path}:{number}")\n        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):\n            raise ValueError(f"Invalid normalized box: {path}:{number}")\n        if min(x-w/2, y-h/2) < -0.011 or max(x+w/2, y+h/2) > 1.011:\n            raise ValueError(f"Box outside image: {path}:{number}")\n        target = mapping[int(cls)]\n        if target is None:\n            ignored += 1\n        else:\n            output.append((target, x, y, w, h))\n    return output, ignored\n\ndef source_group(stem):\n    # Roboflow augmentations retain the pre-.rf. name. This is not a driver ID.\n    return re.split(r"[._]rf[._]", stem, maxsplit=1, flags=re.I)[0]\n\ndef prepare_dataset(raw, output, min_images_per_class=1, *, allowed_invalid_labels=(), progress=None):\n    raw, output = Path(raw).resolve(), Path(output).resolve()\n    if output.exists():\n        raise FileExistsError(f"Output already exists; choose a new folder: {output}")\n    metadata = yaml.safe_load((raw / "data.yaml").read_text(encoding="utf-8-sig"))\n    names = metadata["names"]\n    names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}\n    mapping = {k: ALIASES.get(normalize(v)) for k, v in names.items()}\n    if set(mapping.values()) - {None} != set(range(4)):\n        raise ValueError(f"Source must contain all four classes. Found: {names}")\n    allowed_invalid_labels = set(allowed_invalid_labels)\n    rows, errors, invalid_files = [], [], []\n    started = time.monotonic()\n    def report(message):\n        if progress is not None:\n            progress(message)\n    for split, folder in [("train", "train"), ("val", "valid" if (raw / "valid").is_dir() else "val"), ("test", "test")]:\n        images = sorted(p for p in (raw / folder / "images").rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})\n        if not images:\n            raise ValueError(f"No images in source {split}; do not create a random test set.")\n        report(f"Проверка {split}: {len(images)} изображений")\n        for index, image in enumerate(images, 1):\n            label = raw / folder / "labels" / image.relative_to(raw / folder / "images").with_suffix(".txt")\n            try:\n                invalid = None\n                try:\n                    boxes, ignored = parse_labels(label, mapping)\n                except ValueError as exc:\n                    relative_label = label.relative_to(raw).as_posix()\n                    if relative_label not in allowed_invalid_labels or not str(exc).startswith("Expected detection boxes, 5 fields:"):\n                        raise\n                    invalid = {"image": image.relative_to(raw).as_posix(),\n                               "label": relative_label, "split": split,\n                               "reason": "invalid_detection_label", "error": str(exc),\n                               "label_sha256": sha256(label),\n                               "line_field_counts": [len(line.split()) for line in label.read_text(encoding="utf-8-sig").splitlines() if line.strip()]}\n                    boxes, ignored = [], 0\n                    report(f"Исключён повреждённый пример: {relative_label}")\n                with Image.open(image) as im:\n                    im = im.convert("RGB")\n                    pixels = hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()\n                rows.append(dict(split=split, image=image, label=label, sha=sha256(image),\n                                 pixels=pixels, group=source_group(image.stem), boxes=boxes, ignored=ignored, invalid=invalid))\n                if invalid:\n                    invalid_files.append(invalid)\n            except (ValueError, OSError) as exc:\n                errors.append(str(exc))\n            if index % 250 == 0 or index == len(images):\n                report(f"{split}: {index}/{len(images)} ({index/len(images):.0%}); прошло {time.monotonic()-started:.0f} с")\n    if errors:\n        raise ValueError(f"{len(errors)} invalid files. First errors:\\n" + "\\n".join(errors[:12]))\n    # Quarantine every member of a group touching more than one split.\n    lookup = {kind: defaultdict(set) for kind in ("sha", "pixels", "group")}\n    invalid_keys = {kind: set() for kind in lookup}\n    for row in rows:\n        for kind in lookup:\n            lookup[kind][row[kind]].add(row["split"])\n            if row["invalid"]:\n                invalid_keys[kind].add(row[kind])\n    quarantined, kept, seen = [], [], set()\n    for row in rows:\n        invalid_related = next((kind for kind in lookup if row[kind] in invalid_keys[kind]), None)\n        reason = next((kind for kind in lookup if len(lookup[kind][row[kind]]) > 1), None)\n        if row["invalid"]:\n            quarantined.append(row["invalid"])\n        elif invalid_related:\n            quarantined.append({"image": str(row["image"].relative_to(raw)), "reason": "related_to_invalid_" + invalid_related})\n        elif reason:\n            quarantined.append({"image": str(row["image"].relative_to(raw)), "reason": "cross_split_" + reason})\n        elif (row["split"], row["pixels"]) in seen:\n            quarantined.append({"image": str(row["image"].relative_to(raw)), "reason": "within_split_duplicate"})\n        else:\n            seen.add((row["split"], row["pixels"]))\n            kept.append(row)\n    counts = {}\n    for split in ("train", "val", "test"):\n        items = [r for r in kept if r["split"] == split]\n        per_image = Counter(c for r in items for c in {b[0] for b in r["boxes"]})\n        counts[split] = {"images": len(items), "images_per_class": {n: per_image[i] for i, n in enumerate(CLASSES)},\n                         "boxes_per_class": dict(Counter(CLASSES[b[0]] for r in items for b in r["boxes"])),\n                         "empty_target_labels": sum(not r["boxes"] for r in items)}\n        if any(per_image[c] < min_images_per_class for c in range(4)):\n            raise ValueError(f"Insufficient class coverage after quarantine: {split}: {counts[split]}")\n    output.mkdir(parents=True)\n    report(f"Проверка завершена. Запись {len(kept)} изображений; исключено {len(quarantined)}")\n    manifest = []\n    for index, row in enumerate(kept, 1):\n        stem = hashlib.sha256(str(row["image"].relative_to(raw)).encode()).hexdigest()[:24]\n        image = output / row["split"] / "images" / (stem + row["image"].suffix.lower())\n        label = output / row["split"] / "labels" / (stem + ".txt")\n        image.parent.mkdir(parents=True, exist_ok=True)\n        label.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(row["image"], image)\n        label.write_text("\\n".join(f"{c} {x:.8f} {y:.8f} {w:.8f} {h:.8f}" for c, x, y, w, h in row["boxes"]), encoding="utf-8")\n        manifest.append({"split": row["split"], "source": str(row["image"].relative_to(raw)), "image": str(image.relative_to(output)), "sha256": row["sha"], "group": row["group"]})\n        if index % 500 == 0 or index == len(kept):\n            report(f"Запись: {index}/{len(kept)} ({index/len(kept):.0%})")\n    (output / "data.yaml").write_text(yaml.safe_dump({"path": str(output), "train": "train/images", "val": "val/images", "test": "test/images", "names": dict(enumerate(CLASSES))}, sort_keys=False), encoding="utf-8")\n    audit = {"source": SOURCE, "source_classes": names, "counts": counts, "quarantined": quarantined,\n             "invalid_files": invalid_files, "allowed_invalid_labels": sorted(allowed_invalid_labels),\n             "manifest": manifest, "discarded_non_target_boxes": sum(r["ignored"] for r in rows),\n             "driver_independent_split_verified": False,\n             "annotation_completeness_verified": False,\n             "limitations": ["Source split has no verified driver/session IDs.", "Exact/pixel duplicates and common Roboflow name groups checked; adjacent frames may remain.", "Dropping face/food assumes four target classes were fully annotated; inspect contact sheet.", "An empty target label is not a verified real-world negative."]}\n    (output / "dataset_audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")\n    report(f"Готово за {time.monotonic()-started:.0f} с. Отчёт: {output / \'dataset_audit.json\'}")\n    return audit\n'

import sys, json, shutil, types
from pathlib import Path
from datetime import datetime, timezone
assert 'WORK' in globals() and 'RAW' in globals(), 'Сначала нужны результаты шагов 1 и 2 в этой сессии.'
WORK, RAW = Path(WORK), Path(RAW)
assert (RAW/'data.yaml').is_file(), 'Скачанные данные не найдены. Возможно, среда Colab была сброшена.'
(WORK/'driverguard_data.py').write_text(UTILITY_SOURCE, encoding='utf-8')
# Replace the in-memory module too: re-import alone would reuse the old code.
_dg_module = types.ModuleType('driverguard_data')
_dg_module.__file__ = str(WORK/'driverguard_data.py')
exec(compile(UTILITY_SOURCE, _dg_module.__file__, 'exec'), _dg_module.__dict__)
sys.modules['driverguard_data'] = _dg_module
if str(WORK) not in sys.path: sys.path.insert(0, str(WORK))
from driverguard_data import prepare_dataset, CLASSES, SOURCE, sha256, KNOWN_PFE_V1_INVALID_LABELS
DATA = WORK / ('prepared_fixed_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
print('Исправленная проверка: используем уже скачанные данные.', flush=True)
print('Исходные файлы остаются в:', RAW, flush=True)
audit = prepare_dataset(RAW, DATA,
    allowed_invalid_labels=KNOWN_PFE_V1_INVALID_LABELS,
    progress=lambda message: print(message, flush=True))
print(json.dumps(audit['counts'], ensure_ascii=False, indent=2))
print('Файлов с неправильным форматом:', len(audit['invalid_files']))
print('Всего исключено примеров:', len(audit['quarantined']))
if globals().get('BACKUP') is not None:
    _audit_backup = Path(BACKUP) / DATA.name
    _audit_backup.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DATA/'dataset_audit.json', _audit_backup/'dataset_audit.json')
print('ГОТОВО. Запустите шаг 4 и проверьте картинки с рамками. Затем — шаг 5.')


## 4. Посмотрите разметку перед обучением
У телефона и сигареты рамка должна охватывать **сам предмет**: Python-детектор проверяет его положение около лица.
Проверьте также, что `seatbelt_off` соответствует человеку без ремня, а не просто отсутствию метки `seatbelt_on`.
Если рамки неверны или пропущено много предметов, сначала исправьте разметку: высокий показатель на такой разметке не означает хорошую модель.

In [ ]:
from PIL import Image, ImageDraw
import random
import matplotlib.pyplot as plt
random.seed(42)
all_images = sorted((DATA/'train'/'images').glob('*'))
by_class = {i: [] for i in range(4)}
for p in all_images:
    lines = (DATA/'train'/'labels'/f'{p.stem}.txt').read_text().splitlines()
    for cls in {int(line.split()[0]) for line in lines if line.strip()}:
        by_class[cls].append(p)
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
palette = ['#00AEEF', '#FF9F1C', '#2ECC71', '#F54265']
for cls in range(4):
    sample = random.sample(by_class[cls], min(4, len(by_class[cls])))
    for j in range(4):
        ax = axes[cls, j]; ax.axis('off')
        if j >= len(sample): continue
        p = sample[j]; im = Image.open(p).convert('RGB'); draw = ImageDraw.Draw(im)
        w, h = im.size
        for line in (DATA/'train'/'labels'/f'{p.stem}.txt').read_text().splitlines():
            c, x, y, bw, bh = map(float, line.split()); c = int(c)
            box = [(x-bw/2)*w, (y-bh/2)*h, (x+bw/2)*w, (y+bh/2)*h]
            draw.rectangle(box, outline=palette[c], width=max(2, w//200))
            draw.text((max(0, box[0]), max(0, box[1]-14)), CLASSES[c], fill=palette[c], stroke_width=1, stroke_fill='black')
        ax.imshow(im); ax.set_title(CLASSES[cls], fontsize=12)
plt.tight_layout(); plt.savefig(WORK/'annotation_preview.jpg', dpi=150); plt.show()

## 5. Обучение
Режим `quality`: YOLOv8n, 640 px, до 60 эпох с ранней остановкой. `quick`: 10 эпох для проверки процесса, не окончательная модель.
Продолжительность зависит от GPU и размера набора; ориентируйтесь на ETA после первой эпохи. Не закрывайте сессию до первого сохранения в Drive.
Если Colab оборвётся, `last.pt` находится в папке резервной копии. В новой сессии повторите подготовку данных; затем укажите этот файл в `START_WEIGHTS`, чтобы продолжить дообучение. Это новый запуск, не точное восстановление оптимизатора.

In [ ]:
from ultralytics import YOLO, settings
settings.update({'sync': False})
MODE = 'quality'  # 'quality' или 'quick'
START_WEIGHTS = 'yolov8n.pt'  # либо путь к своему last.pt в Drive
assert MODE in {'quality', 'quick'}, 'MODE: quality или quick'
assert (DATA/'dataset_audit.json').is_file(), 'Сначала завершите шаг 3.'
EPOCHS = 60 if MODE == 'quality' else 10
model = YOLO(START_WEIGHTS)
def report_epoch(trainer):
    now = time.monotonic()
    if not hasattr(trainer, '_dg_started'):
        return
    done = trainer.epoch + 1
    elapsed = now - trainer._dg_started
    eta = elapsed / done * max(0, trainer.epochs - done)
    print(f'Эпоха {done}/{trainer.epochs}; прошло {elapsed/60:.1f} мин; примерно осталось {eta/60:.1f} мин (без экспорта).', flush=True)

def training_started(trainer):
    trainer._dg_started = time.monotonic()

def backup_epoch(trainer):
    if BACKUP is not None:
        for name in ('last.pt', 'best.pt'):
            p = Path(trainer.save_dir)/'weights'/name
            if p.exists():
                temp = BACKUP/(name + '.tmp')
                shutil.copy2(p, temp)
                temp.replace(BACKUP/name)
        for name in ('results.csv', 'args.yaml'):
            p = Path(trainer.save_dir)/name
            if p.exists():
                temp = BACKUP/(name + '.tmp')
                shutil.copy2(p, temp)
                temp.replace(BACKUP/name)
model.add_callback('on_train_start', training_started)
model.add_callback('on_fit_epoch_end', report_epoch)
model.add_callback('on_model_save', backup_epoch)
train_result = model.train(
    data=str(DATA/'data.yaml'), epochs=EPOCHS, imgsz=640, batch=8,
    device=0, workers=2, cache=False, patience=12, seed=42,
    deterministic=True, pretrained=True, amp=True, optimizer='AdamW',
    lr0=0.001, cos_lr=True, close_mosaic=10, fliplr=0.0,
    project=str(WORK/'runs'), name='driverguard', exist_ok=False,
    save=True, plots=True, verbose=False)
TRAIN_DIR = Path(model.trainer.save_dir)
BEST = TRAIN_DIR/'weights'/'best.pt'
assert BEST.is_file(), 'Обучение не сохранило best.pt.'
print('Лучшие веса:', BEST)

## 6. Измерение качества
Вес `best.pt` выбирается обучением по validation. Test используется только после выбора весов.
Отчёт содержит mAP50, mAP50–95, precision и recall по классам. Это метрики изображений исходного набора, не проценты предотвращённых ДТП и не частота ложных тревог за час.

In [ ]:
best = YOLO(str(BEST))
EVAL_DIRS = {}
def measure(split):
    run_name = split + '_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    m = best.val(data=str(DATA/'data.yaml'), split=split, imgsz=640, batch=8,
                 device=0, workers=2, plots=True, project=str(WORK/'evaluation'),
                 name=run_name, exist_ok=True, verbose=False)
    EVAL_DIRS[split] = WORK/'evaluation'/run_name
    per_class = {}
    for j, cls in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(j)
        per_class[CLASSES[int(cls)]] = dict(precision=float(p), recall=float(r), mAP50=float(ap50), mAP50_95=float(ap))
    return dict(mAP50=float(m.box.map50), mAP50_95=float(m.box.map),
                precision=float(m.box.mp), recall=float(m.box.mr), per_class=per_class,
                support=audit['counts'][split])
metrics = {'validation': measure('val'), 'test': measure('test')}
metrics['candidate_screen'] = {
    'mAP50_threshold': 0.85,
    'passed': metrics['test']['mAP50'] >= 0.85 and len(metrics['test']['per_class']) == 4,
    'meaning': 'Initial screening only; not a field approval or deployment gate.'}
(WORK/'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(metrics, ensure_ascii=False, indent=2))

## 7. Экспорт для DriverGuard и проверка совместимости
Формат ONNX: вход 1×3×640×640, четыре класса в правильном порядке, обычный выход YOLOv8 без встроенного NMS.
В ноутбук включён исходный модуль `device/dg/detector.py` из переданного проекта: проверяется загрузка через настоящий класс `OnnxYolo`.

In [ ]:
DETECTOR_SOURCE = '"""Детектор нарушений (DEV-8…DEV-10, ML-4): YOLOv8-ONNX через onnxruntime на CPU в отдельном потоке.\n\nДо своей модели — COCO yolov8n (класс «cell phone»); после `ml/scripts/registry.py promote vN` —\nсвоя модель с классами phone / cigarette / seatbelt_on / seatbelt_off. Главный цикл (MediaPipe, 30 FPS)\nне ждёт детектор: submit() оставляет последний кадр, поток считает не чаще interval_ms, latest() отдаёт\nпоследние сигналы. Телефон и сигарета засчитываются только в зоне «у лица» (Приложение Г: телефон на\nпанели — не событие), поэтому без лица в кадре они равны 0.\n"""\n\nfrom __future__ import annotations\n\nimport ast\nimport json\nimport logging\nimport os\nimport threading\nimport time\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nimport numpy as np\n\nlog = logging.getLogger("dg.detector")\n\nDEVICE_DIR = Path(__file__).resolve().parent.parent\nCLASS_ALIASES: dict[str, set[str]] = {\n    "phone": {"phone", "cell phone", "cellphone", "cell_phone", "mobile phone", "mobile_phone", "smartphone"},\n    "cigarette": {"cigarette", "cigar", "smoking", "vape", "e-cigarette"},\n    "seatbelt_on": {"seatbelt_on", "seatbelt", "seat belt", "seat_belt", "belt_on"},\n    "seatbelt_off": {"seatbelt_off", "no_seatbelt", "no seatbelt", "no seat belt", "belt_off"},\n}\nMAX_VERSION_LEN = 20  # devices.model_version в backend — String(20)\n\n\n@dataclass\nclass Box:\n    cls: str  # наш класс: phone | cigarette | seatbelt_on | seatbelt_off\n    conf: float\n    x1: float  # нормированные координаты 0..1\n    y1: float\n    x2: float\n    y2: float\n\n    @property\n    def center(self) -> tuple[float, float]:\n        return (self.x1 + self.x2) / 2, (self.y1 + self.y2) / 2\n\n\n@dataclass\nclass DetectorSignals:\n    t_ms: int = 0\n    phone_conf: float = 0.0\n    smoke_conf: float = 0.0\n    seatbelt_off_conf: float = 0.0\n    seatbelt_on_conf: float = 0.0\n    boxes: list[Box] = field(default_factory=list)\n\n\n# ── чистая математика (та же, что в ml/dgml/onnx_infer.py) ──\ndef letterbox(img: np.ndarray, size: int, color: int = 114) -> tuple[np.ndarray, float, tuple[int, int]]:\n    import cv2  # slim-образ симулятора без OpenCV импортирует dg.app — cv2 нужен только детектору\n\n    h, w = img.shape[:2]\n    ratio = min(size / h, size / w)\n    nw, nh = int(round(w * ratio)), int(round(h * ratio))\n    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR) if (nw, nh) != (w, h) else img\n    dw, dh = (size - nw) // 2, (size - nh) // 2\n    canvas = np.full((size, size, 3), color, dtype=np.uint8)\n    canvas[dh : dh + nh, dw : dw + nw] = resized\n    return canvas, ratio, (dw, dh)\n\n\ndef nms(boxes: np.ndarray, scores: np.ndarray, iou_thr: float = 0.45) -> list[int]:\n    if len(boxes) == 0:\n        return []\n    x1, y1, x2, y2 = boxes.T\n    areas = (x2 - x1) * (y2 - y1)\n    order = scores.argsort()[::-1]\n    keep: list[int] = []\n    while order.size:\n        i = int(order[0])\n        keep.append(i)\n        if order.size == 1:\n            break\n        rest = order[1:]\n        xx1, yy1 = np.maximum(x1[i], x1[rest]), np.maximum(y1[i], y1[rest])\n        xx2, yy2 = np.minimum(x2[i], x2[rest]), np.minimum(y2[i], y2[rest])\n        inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)\n        order = rest[inter / (areas[i] + areas[rest] - inter + 1e-9) <= iou_thr]\n    return keep\n\n\ndef decode(\n    output: np.ndarray,\n    ratio: float,\n    pad: tuple[int, int],\n    shape: tuple[int, int],\n    conf_thr: float,\n    iou_thr: float,\n    nc: int,\n) -> list[tuple[int, float, float, float, float, float]]:\n    """Выход YOLOv8 (1, 4+nc, N) → [(cls_id, conf, x1, y1, x2, y2)] в пикселях кадра."""\n    out = output[0] if output.ndim == 3 else output\n    if out.shape[0] == 4 + nc:\n        out = out.T\n    scores = out[:, 4:]\n    cls_ids = scores.argmax(axis=1)\n    confs = scores[np.arange(len(cls_ids)), cls_ids]\n    mask = confs >= conf_thr\n    if not mask.any():\n        return []\n    b, c, s = out[mask, :4], cls_ids[mask], confs[mask]\n    xyxy = np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2, b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], 1)\n    h, w = shape\n    dets = []\n    for ci in np.unique(c):\n        idx = np.where(c == ci)[0]\n        for k in nms(xyxy[idx], s[idx], iou_thr):\n            x1, y1, x2, y2 = xyxy[idx[k]]\n            dets.append(\n                (\n                    int(ci),\n                    float(s[idx[k]]),\n                    float(np.clip((x1 - pad[0]) / ratio, 0, w)),\n                    float(np.clip((y1 - pad[1]) / ratio, 0, h)),\n                    float(np.clip((x2 - pad[0]) / ratio, 0, w)),\n                    float(np.clip((y2 - pad[1]) / ratio, 0, h)),\n                )\n            )\n    return dets\n\n\ndef parse_names(meta: dict) -> dict[int, str]:\n    raw = meta.get("names")\n    if not raw:\n        return {}\n    try:\n        names = ast.literal_eval(raw)\n    except (ValueError, SyntaxError):\n        try:\n            names = json.loads(raw)\n        except json.JSONDecodeError:\n            return {}\n    if isinstance(names, list):\n        return dict(enumerate(str(n) for n in names))\n    return {int(k): str(v) for k, v in names.items()}\n\n\ndef map_classes(names: dict[int, str]) -> dict[int, str]:\n    """id класса модели → наш класс; лишние классы (COCO person, car…) не попадают в карту."""\n    out: dict[int, str] = {}\n    for idx, name in names.items():\n        key = name.strip().lower().replace("-", " ")\n        for ours, aliases in CLASS_ALIASES.items():\n            if key == ours or key in aliases or key.replace(" ", "_") in aliases:\n                out[idx] = ours\n                break\n    return out\n\n\ndef near_face(box: Box, face: tuple[float, float, float, float] | None, scale: float) -> bool:\n    """Центр бокса в зоне «у лица»: бокс лица, расширенный в scale раз и сдвинутый вниз (ухо, рот, рука)."""\n    if face is None:\n        return False\n    fx1, fy1, fx2, fy2 = face\n    fw, fh = fx2 - fx1, fy2 - fy1\n    if fw <= 0 or fh <= 0:\n        return False\n    cx, cy = (fx1 + fx2) / 2, (fy1 + fy2) / 2 + fh * 0.25\n    zx1, zx2 = cx - fw * scale / 2, cx + fw * scale / 2\n    zy1, zy2 = cy - fh * scale / 2, cy + fh * scale / 2\n    bx, by = box.center\n    return zx1 <= bx <= zx2 and zy1 <= by <= zy2\n\n\ndef summarize(boxes: list[Box], face: tuple | None, near_face_scale: float, t_ms: int) -> DetectorSignals:\n    sig = DetectorSignals(t_ms=t_ms, boxes=boxes)\n    for b in boxes:\n        if b.cls == "phone" and near_face(b, face, near_face_scale):\n            sig.phone_conf = max(sig.phone_conf, b.conf)\n        elif b.cls == "cigarette" and near_face(b, face, near_face_scale):\n            sig.smoke_conf = max(sig.smoke_conf, b.conf)\n        elif b.cls == "seatbelt_off":\n            sig.seatbelt_off_conf = max(sig.seatbelt_off_conf, b.conf)\n        elif b.cls == "seatbelt_on":\n            sig.seatbelt_on_conf = max(sig.seatbelt_on_conf, b.conf)\n    return sig\n\n\nclass OnnxYolo:\n    """YOLOv8-ONNX через onnxruntime (CPUExecutionProvider). Имена классов и размер входа — из модели."""\n\n    def __init__(self, path: str | Path, conf: float = 0.35, iou: float = 0.45, threads: int = 2):\n        import onnxruntime as ort  # тяжёлый импорт — только здесь\n\n        opts = ort.SessionOptions()\n        opts.intra_op_num_threads = max(1, threads)\n        opts.inter_op_num_threads = 1\n        self.path = Path(path)\n        self.session = ort.InferenceSession(str(path), opts, providers=["CPUExecutionProvider"])\n        inp = self.session.get_inputs()[0]\n        self.input_name = inp.name\n        self.size = int(inp.shape[-1]) if isinstance(inp.shape[-1], int) else 640\n        meta = self.session.get_modelmeta().custom_metadata_map or {}\n        self.names = parse_names(meta)\n        self.class_map = map_classes(self.names)\n        self.version = (meta.get("dg_version") or self.path.stem)[:MAX_VERSION_LEN]\n        self.conf, self.iou = conf, iou\n        if not self.class_map:\n            raise RuntimeError(f"В модели {self.path.name} нет классов phone/cigarette/seatbelt — не та модель")\n\n    def detect(self, frame_bgr: np.ndarray) -> list[Box]:\n        canvas, ratio, pad = letterbox(frame_bgr, self.size)\n        blob = np.ascontiguousarray(canvas[:, :, ::-1].transpose(2, 0, 1)[None].astype(np.float32) / 255.0)\n        output = self.session.run(None, {self.input_name: blob})[0]\n        h, w = frame_bgr.shape[:2]\n        boxes = []\n        for cls_id, conf, x1, y1, x2, y2 in decode(output, ratio, pad, (h, w), self.conf, self.iou, len(self.names)):\n            ours = self.class_map.get(cls_id)\n            if ours:\n                boxes.append(Box(ours, round(conf, 3), x1 / w, y1 / h, x2 / w, y2 / h))\n        return boxes\n\n\nclass ViolationDetector:\n    """Фоновый поток над моделью: последний кадр → сигналы; главный цикл не блокируется."""\n\n    def __init__(self, model, interval_ms: int = 250, near_face_scale: float = 1.8, stale_ms: int = 1500):\n        self.model = model\n        self.interval_ms = interval_ms\n        self.near_face_scale = near_face_scale\n        self.stale_ms = stale_ms\n        self.version = getattr(model, "version", "unknown")\n        self._lock = threading.Lock()\n        self._new = threading.Event()\n        self._stop = threading.Event()\n        self._pending: tuple[np.ndarray, int, tuple | None] | None = None\n        self._latest = DetectorSignals()\n        self._thread = threading.Thread(target=self._run, name="dg-detector", daemon=True)\n        self.stats = {"frames": 0, "infer_ms": 0.0, "errors": 0}\n\n    def start(self) -> None:\n        self._thread.start()\n\n    def stop(self) -> None:\n        self._stop.set()\n        self._new.set()\n        if self._thread.is_alive():\n            self._thread.join(timeout=3)\n\n    def submit(self, frame: np.ndarray, t_ms: int, face_box: tuple | None) -> None:\n        with self._lock:\n            self._pending = (frame, t_ms, face_box)  # только последний кадр; старый — в мусор\n        self._new.set()\n\n    def latest(self, t_ms: int) -> DetectorSignals:\n        with self._lock:\n            sig = self._latest\n        if not sig.boxes and sig.t_ms == 0:\n            return sig\n        return sig if t_ms - sig.t_ms <= self.stale_ms else DetectorSignals(t_ms=sig.t_ms)\n\n    def process(self, frame: np.ndarray, t_ms: int, face_box: tuple | None) -> DetectorSignals:\n        """Синхронный вариант (тесты, прогон видеофайла)."""\n        sig = summarize(self.model.detect(frame), face_box, self.near_face_scale, t_ms)\n        with self._lock:\n            self._latest = sig\n        return sig\n\n    def _run(self) -> None:\n        while not self._stop.is_set():\n            self._new.wait(timeout=1.0)\n            self._new.clear()\n            if self._stop.is_set():\n                break\n            with self._lock:\n                job, self._pending = self._pending, None\n            if job is None:\n                continue\n            frame, t_ms, face_box = job\n            started = time.monotonic()\n            try:\n                self.process(frame.copy(), t_ms, face_box)\n            except Exception as exc:  # noqa: BLE001 — детектор не должен ронять DMS\n                self.stats["errors"] += 1\n                if self.stats["errors"] <= 3:\n                    log.warning("Детектор: %s", exc)\n            took = (time.monotonic() - started) * 1000\n            self.stats["frames"] += 1\n            self.stats["infer_ms"] += (took - self.stats["infer_ms"]) * 0.2\n            rest = self.interval_ms / 1000 - (time.monotonic() - started)\n            if rest > 0:\n                self._stop.wait(rest)\n\n\ndef resolve_model_path(model: str) -> Path:\n    custom = os.environ.get("DG_YOLO_MODEL")\n    path = Path(custom) if custom else Path(model)\n    return path if path.is_absolute() else DEVICE_DIR / path\n\n\ndef load_detector(cfg, mode: str | None = None) -> ViolationDetector | None:\n    """auto — включить, если есть onnxruntime и файл модели; on — обязательно (иначе ошибка); off — нет."""\n    mode = (mode or os.environ.get("DG_DETECTOR") or cfg.detector_enabled or "auto").lower()\n    if mode == "off":\n        log.info("Детектор нарушений выключен (detector.enabled=off)")\n        return None\n    path = resolve_model_path(cfg.detector_model)\n    if not path.is_file():\n        msg = f"Модель детектора не найдена: {path} (ml/scripts/export.py --coco или registry.py promote vN)"\n        if mode == "on":\n            raise RuntimeError(msg)\n        log.info("%s — работаем без детектора телефона/курения/ремня", msg)\n        return None\n    try:\n        model = OnnxYolo(path, conf=cfg.detector_conf, threads=cfg.detector_threads)\n    except ImportError:\n        msg = \'onnxruntime не установлен: pip install -e ".[yolo]"\'\n        if mode == "on":\n            raise RuntimeError(msg) from None\n        log.info("%s — работаем без детектора", msg)\n        return None\n    det = ViolationDetector(model, interval_ms=cfg.detector_interval_ms, near_face_scale=cfg.detector_near_face)\n    log.info(\n        "Детектор нарушений: %s (%s), классы %s, вход %d, каждые %d мс",\n        model.version,\n        path.name,\n        sorted(set(model.class_map.values())),\n        model.size,\n        cfg.detector_interval_ms,\n    )\n    return det\n'
(WORK/'driverguard_detector.py').write_text(DETECTOR_SOURCE, encoding='utf-8')

In [ ]:
import numpy as np, onnx, onnxruntime as ort, cv2
exported = Path(best.export(format='onnx', imgsz=640, batch=1, opset=11,
                          dynamic=False, simplify=False, half=False, nms=False, device='cpu'))
onnx.checker.check_model(str(exported))
session = ort.InferenceSession(str(exported), providers=['CPUExecutionProvider'])
assert session.get_inputs()[0].shape == [1, 3, 640, 640]
names = ast.literal_eval(session.get_modelmeta().custom_metadata_map['names'])
assert [names[i] for i in range(4)] == CLASSES
import importlib
import driverguard_detector as actual_device
actual_device = importlib.reload(actual_device)
device_model = actual_device.OnnxYolo(exported)
samples = sorted((DATA/'test'/'images').glob('*'))[:3]
assert samples, 'В тестовой выборке нет изображений для проверки ONNX.'
raw_shapes, parity_errors = [], []
# Check PyTorch/ONNX raw outputs on real held-out images before packaging.
pt = YOLO(str(BEST)).model.cpu().float().eval()
pt.fuse(verbose=False)
for p in samples:
    bgr = cv2.imread(str(p)); assert bgr is not None
    padded, _, _ = actual_device.letterbox(bgr, 640)
    tensor = np.ascontiguousarray(padded[:,:,::-1].transpose(2,0,1)[None], dtype=np.float32) / 255
    output = session.run(None, {session.get_inputs()[0].name: tensor})[0]
    assert output.ndim == 3 and output.shape[0] == 1 and output.shape[1] == 8
    assert np.isfinite(output).all()
    with torch.no_grad(): reference = pt(torch.from_numpy(tensor))
    if isinstance(reference, (tuple, list)): reference = reference[0]
    reference = reference.detach().numpy()
    assert reference.shape == output.shape
    parity_errors.append(float(np.max(np.abs(reference-output))))
    assert np.allclose(reference, output, rtol=0.001, atol=0.01), 'PyTorch/ONNX outputs differ.'
    raw_shapes.append(list(output.shape))
    boxes = device_model.detect(bgr)
    assert all(box.cls in CLASSES for box in boxes)
compatibility = {'onnx_valid': True, 'device_loader': 'device/dg/detector.py:OnnxYolo',
                 'source_sha256': hashlib.sha256(DETECTOR_SOURCE.encode()).hexdigest(),
                 'images_checked': len(samples), 'raw_shapes': raw_shapes,
                 'pytorch_onnx_max_abs_error': parity_errors,
                 'classes': CLASSES, 'field_validation': 'pending'}
(WORK/'compatibility.json').write_text(json.dumps(compatibility, indent=2), encoding='utf-8')
print(f'ONNX проверен на {len(samples)} изображениях. Загрузка исходным модулем DriverGuard успешна.')

## 8. Скачать результат
Архив содержит веса, модель ONNX, отчёты, графики, сведения об источнике и версии библиотек.
Изображения датасета и секреты в архив не добавляются. Веса — **кандидат** для проверки на своих видео; ноутбук не подменяет действующую модель автоматически.

In [ ]:
import importlib.metadata
BUNDLE = WORK/('DriverGuard_Training_Result_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
BUNDLE.mkdir(exist_ok=False)
shutil.copy2(BEST, BUNDLE/'best.pt')
shutil.copy2(exported, BUNDLE/'driverguard-candidate.onnx')
for name in ['metrics.json', 'compatibility.json', 'annotation_preview.jpg', 'driverguard_data.py']:
    shutil.copy2(WORK/name, BUNDLE/name)
shutil.copy2(DATA/'dataset_audit.json', BUNDLE/'dataset_audit.json')
shutil.copy2(TRAIN_DIR/'results.csv', BUNDLE/'training_results.csv')
for source, prefix in [(TRAIN_DIR, 'train'), (EVAL_DIRS['val'], 'val'), (EVAL_DIRS['test'], 'test')]:
    for p in source.glob('*.png'):
        shutil.copy2(p, BUNDLE/f'{prefix}_{p.name}')
versions = {name: importlib.metadata.version(name) for name in ['ultralytics', 'torch', 'roboflow', 'onnx', 'onnxruntime', 'numpy']}
(BUNDLE/'run.json').write_text(json.dumps({'run_id':RUN_ID, 'mode':MODE, 'versions':versions,
    'python':platform.python_version(), 'gpu':torch.cuda.get_device_name(0),
    'source':SOURCE, 'weights_sha256':sha256(BEST), 'onnx_sha256':sha256(exported),
    'trained':True, 'production_approved':False}, ensure_ascii=False, indent=2), encoding='utf-8')
(BUNDLE/'ATTRIBUTION.txt').write_text(
    'Dataset: PFE by PFE, Roboflow workspace pfe-lcrvh, version 1.\n'
    'https://universe.roboflow.com/pfe-lcrvh/pfe-scklc\n'
    'License listed by uploader: CC BY 4.0 https://creativecommons.org/licenses/by/4.0/\n'
    'Changes: remapped to four DriverGuard classes; excluded duplicates; preserved source splits.\n'
    'Training: Ultralytics YOLOv8n; check AGPL-3.0/Enterprise terms before distribution.\n'
    'https://www.ultralytics.com/license\n', encoding='utf-8')
(BUNDLE/'STATUS.txt').write_text('Обучение завершено. Это кандидат для проверки, а не подтверждённая готовность к рейсам.\n'
    'Сначала проверьте metrics.json, compatibility.json и примеры.\n'
    'Передайте весь ZIP для подключения к DriverGuard. Браузерный MediaPipe этим ONNX не заменяется.\n', encoding='utf-8')
(BUNDLE/'browser-model.json').write_text(json.dumps({'schema':1,'classes':CLASSES,'input_size':640,'sha256':sha256(exported),'run_id':RUN_ID,'status':'candidate'}, indent=2), encoding='utf-8')
archive = shutil.make_archive(str(WORK/'DriverGuard_Training_Result'), 'zip', BUNDLE)
if BACKUP is not None:
    shutil.copy2(archive, BACKUP/Path(archive).name)
from google.colab import files
files.download(archive)
print('Результат:', archive)
if BACKUP: print('Копия в Drive:', BACKUP/Path(archive).name)
print('Передайте ZIP: проверим результаты и подключим модель к камере и проверке видео в браузере.')

## Источники и следующий шаг
- [PFE, исходные данные](https://universe.roboflow.com/pfe-lcrvh/pfe-scklc)
- [Ultralytics YOLOv8](https://docs.ultralytics.com/models/yolov8/), [экспорт](https://docs.ultralytics.com/modes/export/)
- [Roboflow Python SDK](https://github.com/roboflow/roboflow-python)

Чтобы проверить перенос на ваши камеры, нужны отдельные видео людей, отсутствующих в обучении. Снимайте на стоящем автомобиле: обычное моргание, закрытие глаз, телефон у лица / на панели, ремень в двух состояниях. Сцену курения можно имитировать незажжённым предметом и явно отметить это при оценке. Подробная карточка съёмки находится рядом с ноутбуком.